Dependencies

In [ ]:
import os
import re
import json
import time
from google import genai
from pyprojroot import here
from dotenv import load_dotenv
from pyprojroot import here

Load env

In [ ]:
root_dir = here()
load_dotenv(root_dir / ".env")

# API Key
gemini_api_key = os.getenv("MODEL_GEN_GEMINI_API_KEY")

# Path
folder_generation_data_path = str(root_dir / os.getenv("FOLDER_GENERATION_DATA_PATH"))

Model Selection

In [9]:
client = genai.Client(api_key=gemini_api_key)
print("List of models that support generateContent:\n")
for m in client.models.list():
    for action in m.supported_actions:
        if action == "generateContent":
            print(m.name)

daftar_model = ["gemini-2.5-flash", "gemini-2.5-flash-lite", "gemini-3-flash-preview", "gemini-3.1-flash-lite-preview"]
pilihan_model = 2

List of models that support generateContent:

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-max-preview-04-2026
models/d

Generation

In [ ]:
def simpan_artikel(title_article, data_article):
    """
    Fungsi untuk membersihkan nama file dan menyimpannya sebagai .md
    """
    # 1. Bersihkan nama title_article untuk dijadikan nama file (hilangkan spasi jadi underscore)
    # Contoh: "Kondisi Ekonomi" -> "kondisi_ekonomi"
    nama_file_aman = re.sub(r'[^a-zA-Z0-9]', '_', title_article.lower())
    nama_file = f"{nama_file_aman}.json"
    
    # 2. Tentukan lokasi folder penyimpanan (Mundur 1 folder ke 'data')
    # folder_simpan = os.path.abspath(os.path.join(os.getcwd(), "..", "data", "generation"))
    
    # 3. Buat foldernya jika belum ada
    os.makedirs(folder_generation_data_path, exist_ok=True)
    
    # 4. Gabungkan folder dan nama file
    path_lengkap = os.path.join(folder_generation_data_path, nama_file)
    
    # 5. Simpan file-nya (menggunakan encoding utf-8 agar emoji dan karakter khusus aman)
    with open(path_lengkap, "w", encoding="utf-8") as file:
        json.dump(data_article, file, indent=4)
        
    print(f"💾 Artikel berhasil disimpan di: {path_lengkap}")

def generate_artikel_dari_web(payload_dari_frontend):
    """
    Fungsi ini dipanggil HANYA ketika user menekan tombol 'Generate' di Web.
    
    Contoh isi payload_dari_frontend:
    {
        "topik_pilihan": ["Kondisi Ekonomi", "Ekonomi"],
        "keywords_bawaan": ["kondisi ekonomi", "kondisi industri", "ekonomi sirkular"],
        "prompt_tambahan_user": "Fokuskan pada dampak ekonomi sirkular bagi UMKM lokal"
    }
    """
    
    topik_str = " dan ".join(payload_dari_frontend["topik_pilihan"])
    keywords_str = ", ".join(payload_dari_frontend["keywords_bawaan"])
    prompt_user = payload_dari_frontend.get("prompt_tambahan_user", "")
    
    # 1. Rakit Prompt Utama
    prompt_sistem = f"""
    Bertindaklah sebagai Jurnalis atau pembuat artikel yang ahli.
    Tuliskan 1 artikel edukasi berdasarkan informasi berikut:
    - Topik_str Utama: {topik_str}
    - Kata Kunci yang WAJIB dibahas: {keywords_str}
    """
    
    # 2. Suntikkan Request Opsional dari Pengguna (Jika Ada)
    if prompt_user:
        prompt_sistem += f"\n\nINSTRUKSI KHUSUS DARI PENGGUNA:\n{prompt_user}"
        
    prompt_sistem += """\n\n
    ATURAN MUTLAK FORMAT OUTPUT:
    1. Kamu WAJIB mengembalikan jawaban HANYA dalam format JSON yang valid.
    2. Struktur JSON harus memiliki persis 2 key: "title" dan "content".
    3. Key "title" berisi teks judul artikel murni.
    4. Key "content" berisi isi artikel dalam format Markdown Dasar yang sangat disederhanakan dengan jumlah kata minimal sebanyak 2000 kata.  

    ATURAN MARKDOWN UNTUK "content":
    - GUNAKAN **teks** untuk menebalkan poin-poin penting.
    - GUNAKAN # atau ## untuk subjudul.
    - GUNAKAN - untuk list bullet poin biasa.
    - DIIZINKAN menggunakan blockquote (>) MAKSIMAL 1 atau 2 kali saja dalam satu artikel (gunakan HANYA untuk menyorot kutipan tokoh atau kesimpulan super penting).
    - DILARANG KERAS menggunakan nested blockquote (>>).
    - DILARANG KERAS menggunakan list bersarang (nested list).
    """

    print(f"🚀 Menerima request dari Web untuk topik_str: {topik_str}")
    
    try:
        print(f"⏳ Menunggu AI menulis artikel, menggunakan model: {daftar_model[pilihan_model]}")
        # 3. Tembak ke Gemini
        response = client.models.generate_content(
            model=daftar_model[pilihan_model],
            contents=prompt_sistem,
            config={"response_mime_type": "application/json"}
        )

        data_article = json.loads(response.text)
        title_article = data_article["title"]
        content_article = data_article["content"]

        print(f"✅ Berhasil! Judul: {title_article}")

        simpan_artikel(title_article, data_article)
        
        return {
            "code": 200,
            "status": "success",
            "title": title_article,
            "content": content_article
        }
    except Exception as e:
        return {
            "code": 500,
            "status": "error",
            "message": str(e)
        }

In [11]:
# Contoh penggunaan
payload = {
    "topik_pilihan": ["Kondisi Ekonomi", "Ekonomi"],
    "keywords_bawaan": ["kondisi ekonomi", "kondisi industri", "ekonomi sirkular"],
    "prompt_tambahan_user": "Fokuskan pada dampak ekonomi sirkular bagi UMKM lokal"
}

# --- SISTEM RETRY YANG AMAN ---
MAKSIMAL_PERCOBAAN = 5
percobaan = 1
hasil_akhir = None

while percobaan <= MAKSIMAL_PERCOBAAN:
    print(f"🔄 Percobaan ke-{percobaan} dari {MAKSIMAL_PERCOBAAN}...")
    
    hasil_akhir = generate_artikel_dari_web(payload)
    
    # Jika sukses (code 200), langsung keluar dari loop!
    if hasil_akhir["code"] == 200:
        print("🎉 AI berhasil membuat artikel JSON dengan format yang benar!")
        break
        
    # Jika gagal, tampilkan pesan error-nya
    print(f"⚠️ Gagal pada percobaan ke-{percobaan}: {hasil_akhir['message']}")
    
    # Tunggu 10 detik sebelum mencoba lagi (jangan jeda jika ini percobaan terakhir)
    if percobaan < MAKSIMAL_PERCOBAAN:
        print("⏳ AI sedang bingung (atau server penuh). Menunggu 10 detik sebelum re-roll...\n")
        time.sleep(10)
        
    percobaan += 1

# --- PENGECEKAN HASIL AKHIR ---
print("-" * 40)
if hasil_akhir and hasil_akhir["code"] == 200:
    print("✅ PROSES SELESAI DENGAN SUKSES!")
    # Tampilkan hanya judul untuk memastikan
    print(f"Judul Artikel: {hasil_akhir["title"]}")
else:
    print("❌ GAGAL TOTAL. AI terus-menerus memberikan format yang salah, model sedang high demand, atau server sedang down.")
    print("Saran: Coba periksa kembali API Key, koneksi internet, Ganti model karena sedang high demand, atau sederhanakan Prompt-nya.")

🔄 Percobaan ke-1 dari 5...
🚀 Menerima request dari Web untuk topik_str: Kondisi Ekonomi dan Ekonomi
⏳ Menunggu AI menulis artikel, menggunakan model: gemini-3-flash-preview
✅ Berhasil! Judul: Transformasi Ekonomi Sirkular: Strategi Penguatan UMKM Lokal di Tengah Dinamika Kondisi Ekonomi dan Industri Nasional
💾 Artikel berhasil disimpan di: c:\Users\ASUS\Ilmu\Pembelajaran\Proyek\article_generation\data\generation\transformasi_ekonomi_sirkular__strategi_penguatan_umkm_lokal_di_tengah_dinamika_kondisi_ekonomi_dan_industri_nasional.json
🎉 AI berhasil membuat artikel JSON dengan format yang benar!
----------------------------------------
✅ PROSES SELESAI DENGAN SUKSES!
Judul Artikel: Transformasi Ekonomi Sirkular: Strategi Penguatan UMKM Lokal di Tengah Dinamika Kondisi Ekonomi dan Industri Nasional
